# Edge-IIoTset: Silver ve Gold Katman Sunumu
Bu notebook, projemizin **Adım 5** (Veri Kalitesi, Temizleme ve Feature Engineering) aşamasının sonuçlarını hocaya sunmak için hazırlanmıştır.

In [3]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Presentation") \
    .config("spark.jars.packages", "io.delta:delta-core_2.12:2.4.0") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

## 1. Silver Layer (Temizlenmiş Veri)
Bu katmanda ham verideki gereksiz kolonlar (ARP, ICMP) atılmış, Null/Inf değerler doldurulmuş ve duplikat satırlar temizlenmiştir.

In [4]:
silver_df = spark.read.format("delta").load("/opt/bitnami/spark/delta-storage/silver/network_traffic")
print(f"Silver Layer Satır Sayısı: {silver_df.count():,}")
print(f"Silver Layer Kolon Sayısı: {len(silver_df.columns)}")

# Noktalı kolon isimleri nedeniyle PySpark hata vermesin diye etrafına ` (backtick) ekliyoruz
silver_df.select("`frame.time`", "`ip.src_host`", "`ip.dst_host`", "Attack_label", "Attack_type").limit(5).toPandas()

Silver Layer Satır Sayısı: 156,986
Silver Layer Kolon Sayısı: 55


,frame.time,ip.src_host,ip.dst_host,Attack_label,Attack_type
0,0.0,0,0.0,1,MITM
1,0.0,0,0.0,1,MITM
2,0.0,0,0.0,1,MITM
3,0.0,0,0.0,1,MITM
4,0.0,0,0.0,1,MITM


## 2. Gold Layer (Feature Engineering - ML'e Hazır)
Bu katmanda Silver verisi üzerine siber saldırı tespitine yönelik **5 yeni makine öğrenmesi özelliği (feature)** eklenmiştir.

In [5]:
gold_df = spark.read.format("delta").load("/opt/bitnami/spark/delta-storage/gold/ml_ready")
print(f"Gold Layer Satır Sayısı: {gold_df.count():,}")
print(f"Gold Layer Kolon Sayısı: {len(gold_df.columns)}")

Gold Layer Satır Sayısı: 156,986
Gold Layer Kolon Sayısı: 60


### Yeni Üretilen 5 Feature'ın Görünümü
DDoS, Port Scanning ve Botnet tespiti için eklenen istatistiksel özelliklerin normal ve saldırı trafiğindeki karşılıkları:

In [6]:
features = ["traffic_asymmetry_ratio", "pkt_size_cv", "flow_intensity", "iat_regularity", "conn_efficiency"]

# Sadece saldırı (Attack_label == 1) trafiğine ait yeni özelliklerin ilk 10 kaydı
gold_df.select(["Attack_type"] + features).filter("Attack_label == 1").limit(10).toPandas()

,Attack_type,traffic_asymmetry_ratio,pkt_size_cv,flow_intensity,iat_regularity,conn_efficiency
0,MITM,0.0,0.999066,0.0,255.0,0.0
1,MITM,0.0,0.998879,0.0,12.0,0.0
2,MITM,0.0,0.999756,0.0,255.0,0.0
3,MITM,0.0,0.998879,0.0,255.0,0.0
4,MITM,0.0,0.998879,0.0,255.0,0.0
5,MITM,0.0,0.998879,0.0,12.0,0.0
6,MITM,0.0,0.998879,0.0,12.0,0.0
7,MITM,0.0,0.999066,0.0,255.0,0.0
8,MITM,0.0,0.998879,0.0,12.0,0.0
9,MITM,0.0,0.999066,0.0,12.0,0.0
